In [ ]:
%pip install requests lightgbm pandas numpy





In [ ]:
import pandas as pd
import requests, time
from tqdm import tqdm

def safe_get(url, retries=5, timeout=20):
    for _ in range(retries):
        try:
            return requests.get(url, timeout=timeout).json()
        except Exception:
            time.sleep(2)
    return None  # give up

cities = [
"New York","Los Angeles","Chicago","Houston","Phoenix","Philadelphia","San Antonio","San Diego","Dallas","San Jose",
"London","Berlin","Paris","Madrid","Rome","Amsterdam","Vienna","Prague","Warsaw","Dublin",
"Tokyo","Osaka","Nagoya","Fukuoka","Sapporo","Seoul","Busan","Shanghai","Beijing","Shenzhen",
"Delhi","Mumbai","Bangalore","Chennai","Kolkata","Hyderabad","Dhaka","Karachi","Singapore","Bangkok",
"Sydney","Melbourne","Brisbane","Perth","Auckland","Wellington","Toronto","Vancouver","Montreal","Calgary",
"Cairo","Johannesburg","Cape Town","Nairobi","Casablanca","Dubai","Abu Dhabi","Doha","Istanbul","Riyadh",
"Moscow","St Petersburg","Kyiv","Minsk","Stockholm","Oslo","Helsinki","Copenhagen","Zurich","Geneva",
"Sao Paulo","Rio de Janeiro","Buenos Aires","Lima","Bogota","Santiago","Mexico City","Guadalajara","Monterrey","Panama City",
"Hong Kong","Taipei","Kuala Lumpur","Manila","Jakarta","Ho Chi Minh City","Hanoi","Tehran","Baghdad","Jerusalem"
]

data = []
for city in tqdm(cities, desc="Downloading Weather"):
    geo = safe_get(f"https://geocoding-api.open-meteo.com/v1/search?name={city}")
    if not geo or "results" not in geo: 
        print(f"⚠️ Geocode fail: {city}")
        continue
    
    lat, lon = geo["results"][0]["latitude"], geo["results"][0]["longitude"]
    
    url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date=2019-01-01&end_date=2024-12-31&daily=temperature_2m_mean,relative_humidity_2m_mean,windspeed_10m_mean&timezone=auto"
    w = safe_get(url)
    
    if not w or "daily" not in w:
        print(f"⚠️ Weather fail: {city}")
        continue
    
    df = pd.DataFrame(w["daily"])
    df["city"] = city
    data.append(df)
    
    time.sleep(0.25)  # slow down so API doesn't choke

df = pd.concat(data)
df.rename(columns={
    "temperature_2m_mean": "temp",
    "relative_humidity_2m_mean": "humidity",
    "windspeed_10m_mean": "wind"
}, inplace=True)

df.to_csv("../data/weather_100_cities.csv", index=False)
print("✅ Download complete:", df.shape)


⚠️ Weather fail: Amsterdam


⚠️ Weather fail: Istanbul


⚠️ Weather fail: Riyadh


⚠️ Weather fail: Moscow


⚠️ Weather fail: St Petersburg


⚠️ Weather fail: Kyiv


⚠️ Weather fail: Minsk


✅ Download complete: (181936, 5)


In [29]:
import pandas as pd

df = pd.read_csv("../data/weather_100_cities.csv")
df['date'] = pd.to_datetime(df['time'])
df.drop(columns=['time'], inplace=True)

df = df.sort_values(['city', 'date'])
df.head()


,temp,humidity,wind,city,date
120560,21.5,72,10.2,Abu Dhabi,2019-01-01
120561,22.0,72,11.5,Abu Dhabi,2019-01-02
120562,22.8,62,10.4,Abu Dhabi,2019-01-03
120563,21.8,70,17.8,Abu Dhabi,2019-01-04
120564,21.4,61,19.0,Abu Dhabi,2019-01-05


In [30]:
def add_lags(df):
    df = df.copy()
    df['temp_lag1'] = df.groupby('city')['temp'].shift(1)
    df['temp_lag7'] = df.groupby('city')['temp'].shift(7)
    df['temp_lag30'] = df.groupby('city')['temp'].shift(30)
    df['target'] = df.groupby('city')['temp'].shift(-1)
    return df

df = add_lags(df)
df = df.dropna()
df.head()


,temp,humidity,wind,city,date,temp_lag1,temp_lag7,temp_lag30,target
120590,20.8,69,16.5,Abu Dhabi,2019-01-31,21.3,19.7,21.5,21.1
120591,21.1,74,9.2,Abu Dhabi,2019-02-01,20.8,20.5,22.0,22.7
120592,22.7,64,10.7,Abu Dhabi,2019-02-02,21.1,21.6,22.8,22.7
120593,22.7,61,20.6,Abu Dhabi,2019-02-03,22.7,22.0,21.8,20.6
120594,20.6,66,22.5,Abu Dhabi,2019-02-04,22.7,23.0,21.4,19.2


In [31]:
train = df[df['date'] < "2024-01-01"]
valid = df[df['date'] >= "2024-01-01"]

features = ["temp_lag1", "temp_lag7", "temp_lag30", "humidity", "wind"]
target = "target"

X_train, y_train = train[features], train[target]
X_valid, y_valid = valid[features], valid[target]

X_train.shape, X_valid.shape


((149068, 5), (30295, 5))

In [32]:
import lightgbm as lgb

lgb_model = lgb.LGBMRegressor(
    n_estimators=900,
    learning_rate=0.03,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8
)

lgb_model.fit(X_train, y_train)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000473 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1117
[LightGBM] [Info] Number of data points in the train set: 149068, number of used features: 5
[LightGBM] [Info] Start training from score 18.200366


,boosting_type,'gbdt'
,num_leaves,64
,max_depth,-1
,learning_rate,0.03
,n_estimators,900
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [33]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

valid_pred = lgb_model.predict(X_valid)
mae = mean_absolute_error(y_valid, valid_pred)
rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
mae, rmse


(1.6848632149757208, 2.43180870815864)

In [34]:
import datetime

def forecast_city(city, model, days=3):
    df_city = df[df.city == city].sort_values("date").copy()
    last = df_city.iloc[-1]
    
    preds = []
    for i in range(days):
        row = {
            "temp_lag1": last['temp'],
            "temp_lag7": df_city.iloc[-7+i]['temp'] if len(df_city) > 7 else last['temp'],
            "temp_lag30": df_city.iloc[-30+i]['temp'] if len(df_city) > 30 else last['temp'],
            "humidity": last['humidity'],
            "wind": last['wind']
        }
        pred = model.predict(pd.DataFrame([row]))[0]
        preds.append(pred)
        
        new_row = last.copy()
        new_row['temp'] = pred
        df_city.loc[len(df_city)] = new_row
        last = new_row
    
    future_dates = pd.date_range(df_city['date'].max()+pd.Timedelta(days=1), periods=days)
    return pd.DataFrame({"date": future_dates, "forecast_temp": preds})


In [35]:
forecast_city("Delhi", lgb_model, days=3)


,date,forecast_temp
0,2024-12-31,13.003911
1,2025-01-01,13.690894
2,2025-01-02,14.155810


In [36]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 🔹 function: fetch past and current weather
def fetch_live_weather(city, days=60):
    # Get lat/long using geocoding API
    geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1"
    geo = requests.get(geo_url).json()
    lat = geo["results"][0]["latitude"]
    lon = geo["results"][0]["longitude"]

    # Fetch past hourly weather data
    start = (datetime.now() - timedelta(days=days)).strftime("%Y-%m-%d")
    end = datetime.now().strftime("%Y-%m-%d")

    weather_url = (
        "https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={lon}"
        f"&start_date={start}&end_date={end}"
        "&hourly=temperature_2m,relativehumidity_2m,windspeed_10m&timezone=auto"
    )

    data = requests.get(weather_url).json()
    
    df = pd.DataFrame({
        "dt": data["hourly"]["time"],
        "temp": data["hourly"]["temperature_2m"],
        "humidity": data["hourly"]["relativehumidity_2m"],
        "wind_speed": data["hourly"]["windspeed_10m"],
    })
    df["dt"] = pd.to_datetime(df["dt"])
    return df

# 🔹 Create lag features like training
def add_lags(df):
    df = df.copy()
    df["temp_lag1"] = df["temp"].shift(1)
    df["temp_lag7"] = df["temp"].shift(7)
    df["temp_lag30"] = df["temp"].shift(30)
    return df.dropna()

# 🔹 Forecast next 3 days
def forecast_next_3_days(model, city):
    df = fetch_live_weather(city)
    df = add_lags(df)

    last_row = df.iloc[-1].copy()
    future = []

    for i in range(3):
        X = last_row[["temp_lag1","temp_lag7","temp_lag30","humidity","wind_speed"]].values.reshape(1, -1)
        pred = model.predict(X)[0]

        date = (datetime.now() + timedelta(days=i+1)).date()
        future.append((date, pred))

        # update lags for next day prediction
        last_row["temp_lag30"] = last_row["temp_lag7"]
        last_row["temp_lag7"] = last_row["temp_lag1"]
        last_row["temp_lag1"] = pred  # predicted becomes lag1

    return pd.DataFrame(future, columns=["date", "forecast_temp"])


In [37]:
forecast_live = forecast_next_3_days(lgb_model, "New Delhi")
print(forecast_live)


         date  forecast_temp
0  2025-11-05      23.033685
1  2025-11-06      23.063128
2  2025-11-07      23.220194


In [38]:
CITY_COORDS = {
    "New Delhi": (28.6139, 77.2090),
    "Mumbai": (19.0760, 72.8777),
    "Bengaluru": (12.9716, 77.5946),
    "Chennai": (13.0827, 80.2707),
    "Hyderabad": (17.3850, 78.4867),
    "Kolkata": (22.5726, 88.3639),
    "Pune": (18.5204, 73.8567),
    "Ahmedabad": (23.0225, 72.5714),
    "Jaipur": (26.9124, 75.7873),
    "Lucknow": (26.8467, 80.9462),
    "Surat": (21.1702, 72.8311),
    "Kanpur": (26.4499, 80.3319),
    "Nagpur": (21.1458, 79.0882),
    "Patna": (25.5941, 85.1376),
    "Indore": (22.7196, 75.8577),
    "Thane": (19.2183, 72.9781),
    "Bhopal": (23.2599, 77.4126),
    "Visakhapatnam": (17.6868, 83.2185),
    "Vadodara": (22.3072, 73.1812),
    "Nashik": (19.9975, 73.7898),

    # USA
    "New York": (40.7128, -74.0060),
    "Los Angeles": (34.0522, -118.2437),
    "Chicago": (41.8781, -87.6298),
    "Houston": (29.7604, -95.3698),
    "Phoenix": (33.4484, -112.0740),
    "Philadelphia": (39.9526, -75.1652),
    "San Antonio": (29.4241, -98.4936),
    "San Diego": (32.7157, -117.1611),
    "Dallas": (32.7767, -96.7970),
    "San Jose": (37.3382, -121.8863),
    "Austin": (30.2672, -97.7431),
    "Seattle": (47.6062, -122.3321),
    "Denver": (39.7392, -104.9903),
    "Boston": (42.3601, -71.0589),
    "Miami": (25.7617, -80.1918),
    "Atlanta": (33.7490, -84.3880),

    # Europe
    "London": (51.5074, -0.1278),
    "Paris": (48.8566, 2.3522),
    "Berlin": (52.5200, 13.4050),
    "Rome": (41.9028, 12.4964),
    "Madrid": (40.4168, -3.7038),
    "Amsterdam": (52.3676, 4.9041),
    "Vienna": (48.2082, 16.3738),
    "Milan": (45.4642, 9.1900),
    "Munich": (48.1351, 11.5820),
    "Barcelona": (41.3851, 2.1734),
    "Lisbon": (38.7223, -9.1393),
    "Prague": (50.0755, 14.4378),
    "Warsaw": (52.2297, 21.0122),

    # Middle East
    "Dubai": (25.2048, 55.2708),
    "Abu Dhabi": (24.4539, 54.3773),
    "Doha": (25.2854, 51.5310),
    "Riyadh": (24.7136, 46.6753),
    "Jeddah": (21.4858, 39.1925),
    "Kuwait City": (29.3759, 47.9774),

    # China
    "Beijing": (39.9042, 116.4074),
    "Shanghai": (31.2304, 121.4737),
    "Shenzhen": (22.5431, 114.0579),
    "Guangzhou": (23.1291, 113.2644),
    "Chengdu": (30.5728, 104.0668),
    "Wuhan": (30.5928, 114.3055),

    # Japan & Korea
    "Tokyo": (35.6895, 139.6917),
    "Osaka": (34.6937, 135.5023),
    "Seoul": (37.5665, 126.9780),
    "Busan": (35.1796, 129.0756),

    # Southeast Asia
    "Singapore": (1.3521, 103.8198),
    "Bangkok": (13.7563, 100.5018),
    "Jakarta": (6.2088, 106.8456),
    "Kuala Lumpur": (3.1390, 101.6869),
    "Manila": (14.5995, 120.9842),
    "Hanoi": (21.0285, 105.8542),

    # Australia
    "Sydney": ( -33.8688, 151.2093),
    "Melbourne": (-37.8136, 144.9631),
    "Perth": (-31.9523, 115.8613),
    "Brisbane": (-27.4698, 153.0251),

    # Africa
    "Cairo": (30.0444, 31.2357),
    "Lagos": (6.5244, 3.3792),
    "Nairobi": ( -1.2921, 36.8219),
    "Johannesburg": (-26.2041, 28.0473),
    "Cape Town": (-33.9249, 18.4241),

    # South America
    "São Paulo": (-23.5505, -46.6333),
    "Rio de Janeiro": (-22.9068, -43.1729),
    "Buenos Aires": (-34.6037, -58.3816),
    "Santiago": (-33.4489, -70.6693),
    "Lima": (-12.0464, -77.0428),
    "Bogotá": (4.7110, -74.0721),
    "Mexico City": (19.4326, -99.1332),
}


In [39]:
import requests, pandas as pd
from datetime import datetime, timedelta

def hourly_open_meteo(city, days=3):
    # needs CITY_COORDS dict with lat/lon
    if city not in CITY_COORDS:
        raise ValueError(f"City '{city}' not in CITY_COORDS.")
    lat, lon = CITY_COORDS[city]
    start = datetime.now().date()
    end = start + timedelta(days=days)
    url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={lon}"
        f"&hourly=temperature_2m&start_date={start}&end_date={end}&timezone=auto"
    )
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    j = r.json()
    if "hourly" not in j:
        raise RuntimeError("Open-Meteo returned no hourly data.")
    df = pd.DataFrame({
        "time": pd.to_datetime(j["hourly"]["time"]),
        "temp_om": j["hourly"]["temperature_2m"]
    })
    return df

def corrected_high_low(city, model, days=3):
    # 1) get ML daily preds (assumes forecast_next_3_days exists)
    ml_daily = forecast_next_3_days(model, city)  # returns date, forecast_temp
    ml_daily = ml_daily.copy()
    ml_daily['date'] = pd.to_datetime(ml_daily['date']).dt.date
    
    # 2) get Open-Meteo hourly forecast (includes day 0..days)
    hourly = hourly_open_meteo(city, days=days)
    hourly['date'] = hourly['time'].dt.date
    
    results = []
    corrected_hourly_all = []
    
    for _, row in ml_daily.iterrows():
        d = row['date']
        ml_mean = float(row['forecast_temp'])
        
        # take hourly for that date from Open-Meteo
        h = hourly[hourly['date'] == d].copy()
        if h.empty:
            # fallback: if no hourly data for that date, broaden range
            h = hourly[(hourly['date'] >= d) & (hourly['date'] < d + timedelta(days=1))].copy()
        if h.empty:
            # give up for this day
            results.append({"date": d, "high": None, "low": None})
            continue
        
        om_mean = h['temp_om'].mean()
        bias = ml_mean - om_mean
        
        # apply bias correction
        h['temp_corrected'] = h['temp_om'] + bias
        high = float(h['temp_corrected'].max())
        low  = float(h['temp_corrected'].min())
        
        # save
        results.append({"date": d, "high": high, "low": low})
        corrected_hourly_all.append(h[['time','temp_om','temp_corrected']])
    
    corrected_hourly_df = pd.concat(corrected_hourly_all, ignore_index=True) if corrected_hourly_all else pd.DataFrame()
    forecast_df = pd.DataFrame(results)
    return forecast_df, corrected_hourly_df

# Example usage:
city = "Bengaluru"
forecast_df, corrected_hourly = corrected_high_low(city, lgb_model, days=3)
print("Corrected daily high/low:")
print(forecast_df)
print("\nSample corrected hourly rows:")
print(corrected_hourly.head(12))


Corrected daily high/low:
         date       high        low
0  2025-11-05  26.348010  18.648010
1  2025-11-06  25.644618  18.444618
2  2025-11-07  25.806851  19.006851

Sample corrected hourly rows:
                  time  temp_om  temp_corrected
0  2025-11-05 00:00:00     20.2        19.54801
1  2025-11-05 01:00:00     19.9        19.24801
2  2025-11-05 02:00:00     19.4        18.74801
3  2025-11-05 03:00:00     19.5        18.84801
4  2025-11-05 04:00:00     19.6        18.94801
5  2025-11-05 05:00:00     19.4        18.74801
6  2025-11-05 06:00:00     19.3        18.64801
7  2025-11-05 07:00:00     20.3        19.64801
8  2025-11-05 08:00:00     22.1        21.44801
9  2025-11-05 09:00:00     23.9        23.24801
10 2025-11-05 10:00:00     25.0        24.34801
11 2025-11-05 11:00:00     25.7        25.04801
